In [ ]:
from tqdm import tqdm
from torchvision import transforms
from torchvision.ops import batched_nms
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
import os
import json
from datetime import datetime
from pathlib import Path

Image.MAX_IMAGE_PIXELS = None

def detect_defects_pipeline(
    image_path,
    output_base_dir,
    binary_weights_path="model_epoch_19.pth",
    defect_weights_path="best_samsung_model.pth",
    window_size=50,
    stride=10,
    batch_size=2048,
    iou_threshold=0.05,
    conf_threshold=0.95
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    image_name = Path(image_path).stem
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    new_folder_name = f"{timestamp}_{image_name}"
    save_dir = os.path.join(output_base_dir, new_folder_name)

    os.makedirs(save_dir, exist_ok=True)
    json_output_path = os.path.join(save_dir, "defect_results.json")

    binary_model = ArtifactCNN().to(device)
    binary_model.load_state_dict(torch.load(binary_weights_path, map_location=device))
    binary_model.eval()

    defect_model = SamsungDefectDetector(num_classes=5).to(device)
    defect_model.load_state_dict(torch.load(defect_weights_path, map_location=device))
    defect_model.eval()

    binary_transform = transforms.Compose([
        transforms.Resize((50, 50)),
        transforms.ToTensor()
    ])

    DATASET_MEAN = [0.5223926305770874, 0.5005241632461548, 0.49497371912002563]
    DATASET_STD = [0.14497309923171997, 0.14863280951976776, 0.14289118349552155]

    defect_transform = transforms.Compose([
        transforms.Resize((50, 50)),
        transforms.ToTensor(),
        transforms.Normalize(mean=DATASET_MEAN, std=DATASET_STD)
    ])

    classes = ['Broken Line', 'Edge False Color', 'Over Desaturation', 'Saturated False Color', 'Smears']
    color_map = {
        'Broken Line': 'red', 'Edge False Color': 'orange',
        'Over Desaturation': 'yellow', 'Saturated False Color': 'cyan', 'Smears': 'magenta'
    }

    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    all_boxes = []
    all_scores = []
    all_labels = []

    x_steps = len(range(0, width - window_size + 1, stride))
    y_steps = len(range(0, height - window_size + 1, stride))
    total_windows = x_steps * y_steps
    print(f"Total windows: {total_windows} | Batch Size: {batch_size} | Conf >= {conf_threshold} | IoU = {iou_threshold}")

    batch_inputs = []
    batch_coords = []
    batch_raw_crops = []

    with torch.no_grad():
        with tqdm(total=total_windows, desc="Scanning Image") as pbar:
            for y in range(0, height - window_size + 1, stride):
                for x in range(0, width - window_size + 1, stride):

                    crop = image.crop((x, y, x + window_size, y + window_size))
                    batch_raw_crops.append(crop)
                    batch_coords.append((x, y))
                    batch_inputs.append(binary_transform(crop))

                    if len(batch_inputs) == batch_size:
                        bin_tensor = torch.stack(batch_inputs).to(device)
                        bin_out = binary_model(bin_tensor)
                        _, bin_preds = torch.max(bin_out, 1)
                        defective_indices = (bin_preds == 1).nonzero(as_tuple=True)[0]

                        if len(defective_indices) > 0:
                            def_tensors = []
                            def_coords = []
                            for idx in defective_indices:
                                def_tensors.append(defect_transform(batch_raw_crops[idx]))
                                def_coords.append(batch_coords[idx])

                            def_tensor_batch = torch.stack(def_tensors).to(device)
                            def_out = defect_model(def_tensor_batch)

                            def_probs = F.softmax(def_out, dim=1)
                            def_scores, def_preds = torch.max(def_probs, dim=1)

                            for i, pred_idx in enumerate(def_preds):
                                dx, dy = def_coords[i]
                                score = def_scores[i].item()
                                label = pred_idx.item()

                                all_boxes.append([dx, dy, dx + window_size, dy + window_size])
                                all_scores.append(score)
                                all_labels.append(label)

                        batch_inputs.clear()
                        batch_coords.clear()
                        batch_raw_crops.clear()
                        pbar.update(batch_size)

            if len(batch_inputs) > 0:
                bin_tensor = torch.stack(batch_inputs).to(device)
                bin_out = binary_model(bin_tensor)
                _, bin_preds = torch.max(bin_out, 1)
                defective_indices = (bin_preds == 1).nonzero(as_tuple=True)[0]

                if len(defective_indices) > 0:
                    def_tensors = []
                    def_coords = []
                    for idx in defective_indices:
                        def_tensors.append(defect_transform(batch_raw_crops[idx]))
                        def_coords.append(batch_coords[idx])

                    def_tensor_batch = torch.stack(def_tensors).to(device)
                    def_out = defect_model(def_tensor_batch)

                    def_probs = F.softmax(def_out, dim=1)
                    def_scores, def_preds = torch.max(def_probs, dim=1)

                    for i, pred_idx in enumerate(def_preds):
                        dx, dy = def_coords[i]
                        score = def_scores[i].item()
                        label = pred_idx.item()
                        all_boxes.append([dx, dy, dx + window_size, dy + window_size])
                        all_scores.append(score)
                        all_labels.append(label)

                pbar.update(len(batch_inputs))


    defect_map = []

    if len(all_boxes) > 0:
        boxes_tensor = torch.tensor(all_boxes, dtype=torch.float32).to(device)
        scores_tensor = torch.tensor(all_scores, dtype=torch.float32).to(device)
        labels_tensor = torch.tensor(all_labels, dtype=torch.int64).to(device)

        print(f"Found {len(boxes_tensor)} total boxes before Confidence filter.")

        conf_mask = scores_tensor >= conf_threshold
        boxes_tensor = boxes_tensor[conf_mask]
        scores_tensor = scores_tensor[conf_mask]
        labels_tensor = labels_tensor[conf_mask]

        print(f"Remaining {len(boxes_tensor)} boxes after Confidence >= {conf_threshold}. Applying NMS...")

        if len(boxes_tensor) > 0:
            keep_indices = batched_nms(boxes_tensor, scores_tensor, labels_tensor, iou_threshold)

            final_boxes = boxes_tensor[keep_indices].cpu().numpy()
            final_scores = scores_tensor[keep_indices].cpu().numpy()
            final_labels = labels_tensor[keep_indices].cpu().numpy()

            for box, score, label in zip(final_boxes, final_scores, final_labels):
                x1, y1, x2, y2 = box
                w, h = x2 - x1, y2 - y1
                defect_map.append({
                    "x": int(x1),
                    "y": int(y1),
                    "w": int(w),
                    "h": int(h),
                    "confidence": round(float(score), 4),
                    "defect_class": classes[int(label)]
                })

    with open(json_output_path, 'w') as f:
        json.dump(defect_map, f, indent=4)

    print("Drawing annotations...")
    draw = ImageDraw.Draw(image)
    try:
        font = ImageFont.truetype("arial.ttf", 25)
    except IOError:
        font = ImageFont.load_default()

    for defect in defect_map:
        x, y, w, h = defect['x'], defect['y'], defect['w'], defect['h']
        d_class = defect['defect_class']
        conf = defect['confidence']
        color = color_map.get(d_class, 'red')

        draw.rectangle([x, y, x + w, y + h], outline=color, width=3)
        text_y = max(0, y - 25)
        draw.text((x, text_y), f"{d_class} {conf:.2f}", fill=color, font=font)

    annotated_image_path = os.path.join(save_dir, f"annotated_{image_name}.jpg")
    image.save(annotated_image_path, format="JPEG", quality=95)

    print(f"Pipeline finished! Kept {len(defect_map)} defect areas.")
    return defect_map

In [ ]:
image_path = "/home/inonzr/datasets/samsung/TE42@gt (1).bmp"
JSON_path = "/home/inonzr/datasets/test"

In [ ]:
generated_json = detect_defects_pipeline(
    image_path=image_path,
    output_base_dir=JSON_path,
    binary_weights_path="model_epoch_19.pth",
    defect_weights_path="best_samsung_model.pth",
    window_size=50,
    stride=10
)